In [4]:
import duckdb
import pandas as pd
import requests
import os
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("API_KEY")
API_KEY is not None

True

In [5]:
#loading raw file
df=pd.read_csv('data/revenues_per_day_fm.csv')



In [6]:
#file size check
df.shape

(337818, 6)

In [7]:
#first 5 rows
df.head()

,id,date,title,revenue,theaters,distributor
0,8b19ad43-3a7e-b14b-49e9-1f7a0eb1568e,2004-09-20,Sky Captain and the World of Tomorrow,925482,3170.0,Paramount Pictures
1,481fc700-fcdd-1919-c53c-09fcd423a596,2004-09-20,Resident Evil: Apocalypse,643680,3284.0,Screen Gems
2,06719cc2-c05e-8c0b-56b6-b8b13e056509,2004-09-20,Mr 3000,425375,2736.0,Walt Disney Studios Motion Pictures
3,3be8e4a8-2716-00be-80be-eb342ba3cbe7,2004-09-20,Wimbledon,416970,2034.0,Universal Pictures
4,c5f26b59-6a64-ae8e-8f58-7cefb09eadfe,2004-09-20,Cellular,412000,2749.0,New Line Cinema


In [8]:
df.describe()

,revenue,theaters
count,3.378180e+05,337657.000000
mean,6.071642e+05,848.611597
std,2.323203e+06,1137.442240
min,0.000000e+00,1.000000
25%,6.592000e+03,45.000000
50%,4.077500e+04,226.000000
75%,3.029205e+05,1402.000000
max,1.574616e+08,4931.000000


In [9]:
df.dtypes

id                 str
date               str
title              str
revenue          int64
theaters       float64
distributor        str
dtype: object

In [10]:
#how many nulls by columns
df.isna().sum()

#0.05% nulls in theaters column, and 0.07% of nulls in distributor column.
#edit: after correctly labeling nulls in distributor columns it's 2,3%

id               0
date             0
title            0
revenue          0
theaters       161
distributor    225
dtype: int64

In [11]:
#fully duplicated rows
df.duplicated().sum()

np.int64(0)

In [12]:
#errors in data type change
df["date"].isna().sum()    


np.int64(0)

In [13]:
#connecting duckdb and creating table for sql
con = duckdb.connect('data/movies.duckdb')
con.sql("CREATE OR REPLACE TABLE raw_revenues AS SELECT * FROM read_csv_auto('data/revenues_per_day_fm.csv')")

In [14]:
con.sql("DESCRIBE raw_revenues")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ title       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ revenue     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ theaters    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ distributor │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [15]:
#how many rows (should be the same as in pandas)
con.sql("SELECT COUNT(*) FROM raw_revenues")


┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       337818 │
└──────────────┘

In [16]:
con.sql("SELECT MIN(date), MAX(date) FROM raw_revenues")

┌────────────┬────────────┐
│ min(date)  │ max(date)  │
│    date    │    date    │
├────────────┼────────────┤
│ 2000-01-01 │ 2023-03-06 │
└────────────┴────────────┘

In [17]:
#how many unique titles
con.sql("SELECT COUNT(DISTINCT(title)) FROM raw_revenues")

┌───────────────────────┐
│ count(DISTINCT title) │
│         int64         │
├───────────────────────┤
│                  6545 │
└───────────────────────┘

In [18]:
con.sql("SELECT COUNT(id) FROM raw_revenues GROUP BY id HAVING COUNT(id)>1")

┌───────────┐
│ count(id) │
│   int64   │
└───────────┘
   0 rows  

In [19]:
con.sql("SELECT title, date, COUNT(*) FROM raw_revenues GROUP BY title, date HAVING COUNT(*)>1")

┌─────────┬──────┬──────────────┐
│  title  │ date │ count_star() │
│ varchar │ date │    int64     │
└─────────┴──────┴──────────────┘
             0 rows            

In [20]:
con.sql("SELECT id, title, revenue FROM raw_revenues WHERE revenue <= 0 ORDER BY revenue")

┌──────────────────────────────────────┬───────────────────────────────────────────┬─────────┐
│                  id                  │                   title                   │ revenue │
│               varchar                │                  varchar                  │  int64  │
├──────────────────────────────────────┼───────────────────────────────────────────┼─────────┤
│ 4cd24fa3-e62d-ded1-cd24-035a3afa4b01 │ White Snake                               │       0 │
│ b36b7203-7ed6-9673-0be7-a272b9343c62 │ Le Cercle Rouge2021 Re-release            │       0 │
│ b54493e6-4d31-1f1b-45ea-e7855d9e3c60 │ The Times of Bill Cunningham              │       0 │
│ e8396d5b-7ae8-5559-ae54-c8445c696014 │ Incitement                                │       0 │
│ 08af2a23-92ea-a3e0-0a26-4b759cc301c4 │ Counter Column                            │       0 │
│ 89bc5398-9498-e752-e058-bbb0fa09bcdd │ The Gentlemen                             │       0 │
│ e2f0dba7-f4d8-0ee4-926b-f0595d6d6a43 │ Incitemen

In [21]:
con.sql("SELECT id, date, title, revenue FROM raw_revenues ORDER BY revenue DESC LIMIT 5")
#Avengers: Endgame earned 157mln$ which is true, so revenue shows in usd.
#This also means that there's 63 rows of movies earning 0 usd in the record.

┌──────────────────────────────────────┬────────────┬────────────────────────────────────────────┬───────────┐
│                  id                  │    date    │                   title                    │  revenue  │
│               varchar                │    date    │                  varchar                   │   int64   │
├──────────────────────────────────────┼────────────┼────────────────────────────────────────────┼───────────┤
│ c5477391-de7d-0cb3-4738-bb9e971d6b53 │ 2019-04-26 │ Avengers: Endgame                          │ 157461641 │
│ e5f9c957-c088-7070-b32c-4e949c0c7d3d │ 2021-12-17 │ Spider-Man: No Way Home                    │ 121850000 │
│ 03ec9798-2fb1-b3e9-b711-3f0a6932f82f │ 2015-12-18 │ Star Wars: Episode VII - The Force Awakens │ 119119282 │
│ c88d93b6-2571-0f47-6e89-e40b0835361a │ 2019-04-27 │ Avengers: Endgame                          │ 109264122 │
│ b9a734bf-fe3f-42cc-404e-8cade0cfc075 │ 2018-04-27 │ Avengers: Infinity War                     │ 106334939 │
└

In [22]:
con.sql("SELECT COUNT(DISTINCT distributor) AS cdis, title FROM raw_revenues GROUP BY title HAVING cdis>1")

┌───────┬─────────────────────────────┐
│ cdis  │            title            │
│ int64 │           varchar           │
├───────┼─────────────────────────────┤
│     2 │ Amazing Grace               │
│     2 │ The Lost City               │
│     2 │ No Escape                   │
│     2 │ Walking with Herb           │
│     2 │ Mission Stories             │
│     2 │ Jujutsu Kaisen 0: The Movie │
│     2 │ The Kid                     │
│     2 │ Serenity                    │
│     2 │ Infamous                    │
│     2 │ Infinity Pool               │
│     · │     ·                       │
│     · │     ·                       │
│     · │     ·                       │
│     2 │ The Rescue                  │
│     2 │ The Proposal                │
│     2 │ Stardust                    │
│     2 │ Les Misu00e9rables          │
│     2 │ Jathi Ratnalu               │
│     2 │ The Dissident               │
│     2 │ Men                         │
│     2 │ The Eye                     │


In [23]:
con.sql("SELECT distributor, COUNT(*) FROM raw_revenues GROUP BY distributor ORDER BY COUNT(*) ")

┌─────────────────────────────────────┬──────────────┐
│             distributor             │ count_star() │
│               varchar               │    int64     │
├─────────────────────────────────────┼──────────────┤
│ Avenue Pictures Productions         │            1 │
│ Five Sisters Productions            │            1 │
│ Artsploitation Films                │            1 │
│ Flyhigh Cinemas                     │            1 │
│ Madman Entertainment                │            1 │
│ Big World Pictures                  │            1 │
│ PBS Distribution                    │            1 │
│ Utopia                              │            1 │
│ Universal Home Video                │            1 │
│ Amplify                             │            1 │
│    ·                                │            · │
│    ·                                │            · │
│    ·                                │            · │
│ The Weinstein Company               │        10120 │
│ Roadside

In [24]:
con.sql("SELECT title, MIN(date), MAX(date), MAX(date)-MIN(date) AS diff FROM raw_revenues GROUP BY title ORDER BY diff DESC")

┌────────────────────────────────────────────┬────────────┬────────────┬───────┐
│                   title                    │ min(date)  │ max(date)  │ diff  │
│                  varchar                   │    date    │    date    │ int64 │
├────────────────────────────────────────────┼────────────┼────────────┼───────┤
│ Charlie's Angels                           │ 2000-11-03 │ 2020-01-16 │  7013 │
│ The Kid                                    │ 2000-07-07 │ 2019-04-18 │  6859 │
│ Pinocchio                                  │ 2002-12-25 │ 2021-03-04 │  6644 │
│ The Lost City                              │ 2006-04-28 │ 2022-06-23 │  5900 │
│ The Grudge                                 │ 2004-10-22 │ 2020-02-27 │  5606 │
│ A Grin Without A Cat                       │ 2002-05-01 │ 2017-07-19 │  5558 │
│ Hellboy                                    │ 2004-04-02 │ 2019-05-16 │  5522 │
│ Confetti                                   │ 2006-09-15 │ 2021-09-16 │  5480 │
│ City of Ghosts            

In [25]:
con.sql("SELECT id, title, date FROM raw_revenues WHERE title IN ('Charlie''s Angels', 'The Kid', 'Pinocchio')")
#id is unique for title and day, not just for distinct movies.

┌──────────────────────────────────────┬──────────────────┬────────────┐
│                  id                  │      title       │    date    │
│               varchar                │     varchar      │    date    │
├──────────────────────────────────────┼──────────────────┼────────────┤
│ f404b371-90b2-826b-6514-e6312d0a9f8d │ Charlie's Angels │ 2000-12-29 │
│ 3a733fdb-c80f-55e1-c90e-09a6c9e0ec07 │ The Kid          │ 2000-08-08 │
│ b57cfc95-88de-90fb-b972-f078ab08de22 │ The Kid          │ 2019-03-10 │
│ 936b83fb-eef3-b64a-b041-3a993162d5f0 │ Charlie's Angels │ 2019-11-23 │
│ d189d53c-6597-94f1-200c-0d723e9a234f │ Charlie's Angels │ 2020-01-13 │
│ e8663642-f860-0a7a-f1eb-0e6bf3b617ac │ The Kid          │ 2000-07-30 │
│ 49f9715a-19a4-c684-146e-493dd42fc22e │ Charlie's Angels │ 2000-12-27 │
│ d582dc1c-e0e4-27c2-e03c-43be3f5d96bc │ Charlie's Angels │ 2019-12-27 │
│ 536fc148-d9ee-7d45-0113-f00fb8be4e5c │ The Kid          │ 2000-08-29 │
│ d12c840e-0220-06dd-3bf0-de88370070cc │ Charlie's 

In [26]:
#some movies are remakes, and some are re-releases. In this CTE assumption is if there is no previous date (null), or date difference is larger than a year it means screening ended and we treat it as another movie. It's a hypothesis and we'll check it with OMDb.
con.sql("""
WITH lag_date AS(
    SELECT id, title, date, LAG(date) OVER (PARTITION BY title ORDER BY date) AS prev_date FROM raw_revenues
),
date_diff AS (
SELECT 
    id, 
    title, 
    date, 
    prev_date, 
    (date)-(prev_date) AS days_between, 
    CASE WHEN prev_date IS NULL THEN 1
        WHEN date_diff('day', prev_date, date) > 365 THEN 1
        ELSE 0
        END AS new_series
FROM lag_date),
series_number AS (
SELECT
    *,
    SUM(new_series) OVER (PARTITION BY title ORDER BY date) AS series_number
FROM date_diff
)
SELECT title, series_number, MIN(date), MAX(date), year(MIN(date)), COUNT(*) AS count FROM series_number GROUP BY title, series_number  ORDER BY title, series_number


""")

┌────────────────────────────────────────────┬───────────────┬────────────┬────────────┬───────────────────┬───────┐
│                   title                    │ series_number │ min(date)  │ max(date)  │ "year"(min(date)) │ count │
│                  varchar                   │    int128     │    date    │    date    │       int64       │ int64 │
├────────────────────────────────────────────┼───────────────┼────────────┼────────────┼───────────────────┼───────┤
│ '71                                        │             1 │ 2015-02-27 │ 2015-05-07 │              2015 │    70 │
│ '83                                        │             1 │ 2021-12-24 │ 2022-01-16 │              2021 │    14 │
│ '85: The Greatest Team in Football History │             1 │ 2018-01-29 │ 2018-02-07 │              2018 │     2 │
│ 10 Cloverfield Lane                        │             1 │ 2016-03-11 │ 2016-06-02 │              2016 │    84 │
│ 10 Years                                   │             1 │ 2

In [27]:
#connecting to database

In [28]:
resp = requests.get("https://www.omdbapi.com/",
                    params={"apikey": API_KEY, "t": "The Kid"})
data = resp.json()
data

{'Title': 'The Kid',
 'Year': '1921',
 'Rated': 'Passed',
 'Released': '06 Feb 1921',
 'Runtime': '68 min',
 'Genre': 'Comedy, Drama, Family',
 'Director': 'Charles Chaplin',
 'Writer': 'Charles Chaplin',
 'Actors': 'Charles Chaplin, Edna Purviance, Jackie Coogan',
 'Plot': 'The Tramp cares for an abandoned child, but events put their relationship in jeopardy.',
 'Language': 'None, English',
 'Country': 'United States',
 'Awards': '2 wins total',
 'Poster': 'https://m.media-amazon.com/images/M/MV5BZGVmZGM0YTUtOTA1ZC00ZDE4LWE4ZDUtMjAxOTk5YjkwN2Q2XkEyXkFqcGc@._V1_SX300.jpg',
 'Ratings': [{'Source': 'Internet Movie Database', 'Value': '8.2/10'},
  {'Source': 'Rotten Tomatoes', 'Value': '100%'}],
 'Metascore': 'N/A',
 'imdbRating': '8.2',
 'imdbVotes': '144,275',
 'imdbID': 'tt0012349',
 'Type': 'movie',
 'DVD': 'N/A',
 'BoxOffice': 'N/A',
 'Production': 'N/A',
 'Website': 'N/A',
 'Response': 'True'}

In [29]:
#year is important for search (many same titles)
#all information is in type string, so there's a need of casting appropriate type in some fields.

In [30]:
#########################
#PROTOTYPE PIPELINE
######################

In [31]:
#Some titles are labeled with year and "Re-release" or "Anniversary" and there would be a problem in next stage
# I need to find as many as I can first

In [32]:
con.sql("""
    SELECT
    title,
    trim(regexp_replace(title,
        '(\d{4}\s*((3D/IMAX|IMAX|3D) Release|[Rr]e-release|release)(\s*\(.*\))?|\d+(st|nd|rd|th) Anniversary.*|Re-release|Recut Release|Limited Domestic Release)$',
        '')) AS base_title
    FROM raw_revenues
    WHERE title ILIKE '%release%'
""")

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/var/folders/7w/89t_pkpx3lz16gymgp29ygnm0000gn/T/ipykernel_42022/1742333209.py:1: SyntaxWarning: invalid escape sequence '\d'
  con.sql("""


┌─────────────────────────────────────────────────────────────┬──────────────────────────────────────────────┐
│                            title                            │                  base_title                  │
│                           varchar                           │                   varchar                    │
├─────────────────────────────────────────────────────────────┼──────────────────────────────────────────────┤
│ The Polar Express2005 IMAX Release                          │ The Polar Express                            │
│ Day Watch2007 Re-release                                    │ Day Watch                                    │
│ Blade RunnerThe Final Cut (2007 re-release)                 │ Blade RunnerThe Final Cut (2007 re-release)  │
│ Vertigo2018 Re-release                                      │ Vertigo                                      │
│ Goldstone2018 Re-release                                    │ Goldstone                                    │
│

In [33]:
con.sql("""CREATE OR REPLACE TABLE stg_revenue AS
WITH lag_date AS(
    SELECT 
        id, 
        title,
        revenue,
        theaters,
        distributor, 
        date, 
        LAG(date) OVER (PARTITION BY title ORDER BY date) AS prev_date 
    FROM raw_revenues
),
date_diff AS (
    SELECT 
        id, 
        title,
        revenue,
        theaters,
        distributor, 
        date, 
        prev_date, 
        (date)-(prev_date) AS days_between, 
        CASE WHEN prev_date IS NULL THEN 1
            WHEN date_diff('day', prev_date, date) > 365 THEN 1
            ELSE 0
        END AS new_series
FROM lag_date),
series_number AS (
    SELECT
        *,
        SUM(new_series) OVER (PARTITION BY title ORDER BY date) AS series_number
    FROM date_diff
)
    SELECT
        id,
        date, 
        title,
        trim(regexp_replace(title,
                '(\d{4}\s*((3D/IMAX|IMAX|3D) Release|[Rr]e-release|release)(\s*\(.*\))?|\d+(st|nd|rd|th) Anniversary.*|Re-release|Recut Release|Limited Domestic Release)$',
                '')) AS base_title,
        revenue,
        theaters,
        series_number,
        CASE WHEN distributor = '-' THEN NULL
            ELSE distributor
            END AS distributor
    FROM series_number 
    

""")

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/var/folders/7w/89t_pkpx3lz16gymgp29ygnm0000gn/T/ipykernel_42022/3515567300.py:1: SyntaxWarning: invalid escape sequence '\d'
  con.sql("""CREATE OR REPLACE TABLE stg_revenue AS


In [34]:
con.sql("SELECT COUNT(*) FROM stg_revenue")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       337818 │
└──────────────┘

In [35]:
con.sql("SELECT * FROM stg_revenue LIMIT 5")

┌──────────────────────────────────────┬────────────┬───────────────────┬───────────────────┬─────────┬──────────┬───────────────┬──────────────────┐
│                  id                  │    date    │       title       │    base_title     │ revenue │ theaters │ series_number │   distributor    │
│               varchar                │    date    │      varchar      │      varchar      │  int64  │  int64   │    int128     │     varchar      │
├──────────────────────────────────────┼────────────┼───────────────────┼───────────────────┼─────────┼──────────┼───────────────┼──────────────────┤
│ d1950dfc-bbf7-8f7d-ab8c-c97eda67cb40 │ 2011-04-07 │ Kill the Irishman │ Kill the Irishman │   10274 │       56 │             1 │ Anchor Bay Films │
│ 51340220-a5fe-c017-b0b0-b83fc767207b │ 2011-04-08 │ Kill the Irishman │ Kill the Irishman │   22610 │       52 │             1 │ Anchor Bay Films │
│ f2f2501d-86b9-90a3-d5ad-d413950d49f5 │ 2011-04-09 │ Kill the Irishman │ Kill the Irishman │   4305

In [38]:
con.sql("""
    SELECT title 
    FROM stg_revenue 
    WHERE title ILIKE '%release%'
        OR title ILIKE '%anniversary%'
    GROUP BY title
    """)

┌──────────────────────────────────────────────────────┐
│                        title                         │
│                       varchar                        │
├──────────────────────────────────────────────────────┤
│ When Harry Met Sally...30th Anniversary              │
│ White Wedding2010 Re-release                         │
│ Cold War2018 Re-release                              │
│ Free Trip to Egypt2019 Re-release                    │
│ Is Genesis History?2018 Re-release                   │
│ Je t'aime, je t'aime2014 Re-release                  │
│ Arctic2019 Re-release                                │
│ Avatar2022  Re-release                               │
│ Bully2012 Re-release                                 │
│ A Man Vanishes2012 Re-release                        │
│           ·                                          │
│           ·                                          │
│           ·                                          │
│ Batman2019 Re-release        

In [39]:
#there's 341 movies with "release" or "anniversary" in their name
#labels are "year+ re-relase" or "release" at the end, or 

In [ ]:
con.sql("""CREATE OR REPLACE TABLE stg_series AS
    SELECT 
        title, 
        base_title, 
        series_number,
        year(min(date)) AS release_year,
        MIN(date) AS first_date,
        MAX(date) AS last_date,
        CASE WHEN title = base_title THEN 0
            ELSE 1
            END AS is_rerelease,
        SUM(revenue) AS sum_revenue,
        COUNT(*) AS days_in_series
    FROM stg_revenue
    GROUP BY title, base_title, series_number


""")

In [37]:
con.sql("SELECT COUNT(*) FROM stg_series")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         6634 │
└──────────────┘

In [40]:
con.sql("""
    SELECT * FROM stg_revenue WHERE distributor = '-'

""")

┌─────────┬──────┬─────────┬────────────┬─────────┬──────────┬───────────────┬─────────────┐
│   id    │ date │  title  │ base_title │ revenue │ theaters │ series_number │ distributor │
│ varchar │ date │ varchar │  varchar   │  int64  │  int64   │    int128     │   varchar   │
└─────────┴──────┴─────────┴────────────┴─────────┴──────────┴───────────────┴─────────────┘
                                           0 rows                                         

In [41]:
con.sql("""
    SELECT COUNT(*) FROM stg_series

""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         6634 │
└──────────────┘

In [42]:
con.sql("""
    SELECT * FROM stg_series WHERE title = 'The Kid'

""")

┌─────────┬────────────┬───────────────┬──────────────┬────────────┬────────────┬──────────────┬─────────────┬────────────────┐
│  title  │ base_title │ series_number │ release_year │ first_date │ last_date  │ is_rerelease │ sum_revenue │ days_in_series │
│ varchar │  varchar   │    int128     │    int64     │    date    │    date    │    int32     │   int128    │     int64      │
├─────────┼────────────┼───────────────┼──────────────┼────────────┼────────────┼──────────────┼─────────────┼────────────────┤
│ The Kid │ The Kid    │             2 │         2019 │ 2019-03-08 │ 2019-04-18 │            0 │     1508593 │             42 │
│ The Kid │ The Kid    │             1 │         2000 │ 2000-07-07 │ 2000-08-31 │            0 │    63550582 │             50 │
└─────────┴────────────┴───────────────┴──────────────┴────────────┴────────────┴──────────────┴─────────────┴────────────────┘

In [45]:
con.sql("""
    SELECT SUM(revenue) FROM stg_revenue

""")

┌──────────────┐
│ sum(revenue) │
│    int128    │
├──────────────┤
│ 205110995141 │
└──────────────┘

In [47]:
con.sql("""
    SELECT SUM(sum_revenue) FROM stg_series

""")

┌──────────────────┐
│ sum(sum_revenue) │
│      int128      │
├──────────────────┤
│     205110995141 │
└──────────────────┘

In [ ]:
#sum of revenues should be the same in both tables (stg_revenue and stg_series)